In [1]:
import os
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import math
from scipy.stats import kendalltau, spearmanr

# ============ 配置 ============
client = OpenAI(
    base_url="http://api.llm.apps.os.dcs.gla.ac.uk/v1",
    api_key='ida_TbtBzRyj8HeV7kmxkeRsgImHnYelRIUMQ3vW9VIf'
)

MODEL = 'qwen-2.5-72b-instruct'

# ============ Few-Shot Prompt (使用你的5个例子) ============
FEW_SHOT_PROMPT_TEMPLATE = """You are an information retrieval expert. Determine if Pseudo-Relevance Feedback (PRF) will benefit or hurt this query.

Here are reference examples:

Example 1 - PRF will BENEFIT
Query: "definition of a sigmet"
Top Documents: [1-4] All consistently define SIGMET as aviation weather advisory with shared terminology (meteorological, aircraft, turbulence, icing). [5] False match (medical "sigmoid") but ranked low.
Analysis: Clear query intent. High-quality consistent results. PRF can extract aviation/meteorological terms to find comprehensive explanations.
Answer: B

Example 2 - PRF will BENEFIT
Query: "who is robert gray"
Top Documents: [1-2] Historical explorer Robert Gray. [3] Modern politician (same name). [4-5] Wrong first names.
Analysis: Despite ambiguity, historical figure dominates top results. PRF can extract biographical terms (captain, explorer, Columbia River) to strengthen historical content and disambiguate.
Answer: B

Example 3 - PRF will HURT
Query: "what is theraderm used for"
Top Documents: [1-2] Theraderm skincare. [3-5] Completely unrelated (thermopile, theraband, thermometer) - spelling similarity only.
Analysis: Catastrophic topic drift. Fragmented results from disparate domains. PRF would mix skincare + physics + fitness terminology, causing massive query drift.
Answer: H

Example 4 - PRF will HURT
Query: "causes of military suicide"
Top Documents: All report statistics/rankings (leading cause, death rates) but don't explain WHY (psychological factors, PTSD, risk factors).
Analysis: Query asks "causes" but results provide "statistics". PRF would extract statistical terms, reinforcing wrong focus rather than pivoting to causal explanations.
Answer: H

Example 5 - PRF will HURT
Query: "what is famvir prescribed for"
Top Documents: All consistently list exact medical indications (genital herpes, cold sores, shingles, chickenpox). Highly authoritative and complete.
Analysis: User's need fully satisfied. PRF provides little benefit and risks introducing noise (side effects, dosage) that dilutes high precision.
Answer: H

Now analyze this query:

Query: "{query}"
Retrieved Documents:
{documents}

Consider: 1) Query intent clarity, 2) Result quality/consistency, 3) Topic coherence, 4) Query drift risk

Answer with ONLY ONE LETTER: B (Benefit) or H (Hurt)
"""

# ============ 加载数据 ============
print("=" * 80)
print("Few-Shot PRF Prediction with Qwen-2.5-72B")
print("=" * 80)

print("\n1. 加载数据...")
df_preference = pd.read_csv('preference.csv')
df_queries = pd.read_csv('result_with_text.csv')
df_colbert = pd.read_csv('df_colbert_deduped.csv')

if 'query_text' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query_text'].first().to_dict()
elif 'query' in df_queries.columns:
    qid_to_query = df_queries.groupby('qid')['query'].first().to_dict()

eval_df = df_preference[df_preference['b_preference_ratio'] > 0].copy()
print(f"✓ 加载了 {len(eval_df)} 个有效查询")

# ============ 预测函数 ============
def predict_few_shot(qid, query, top5_docs, verbose=False):
    """Few-shot prediction with B/H format"""
    
    docs_text = "\n".join([
        f"[{i+1}] [Score: {row['score']:.2f}] {str(row['passage_text'])[:180]}..."
        for i, (_, row) in enumerate(top5_docs.iterrows())
    ])
    
    prompt = FEW_SHOT_PROMPT_TEMPLATE.format(
        query=query,
        documents=docs_text
    )
    
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are an IR expert. Output only B or H."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=1,
            logprobs=True,
            top_logprobs=5
        )
        
        output_letter = response.choices[0].message.content.strip().upper()
        
        if not response.choices[0].logprobs or not response.choices[0].logprobs.content:
            return {
                "prediction": "Benefit" if output_letter == 'B' else "Hurt",
                "confidence": 0.5,
                "success": False
            }
        
        first_token_logprobs = response.choices[0].logprobs.content[0].top_logprobs
        
        b_prob = 0.0
        h_prob = 0.0
        raw_probs = {}
        
        for item in first_token_logprobs:
            token = item.token.strip().upper()
            prob = math.exp(item.logprob)
            raw_probs[token] = prob
            
            if token == 'B':
                b_prob = prob
            elif token == 'H':
                h_prob = prob
        
        total = b_prob + h_prob
        confidence = b_prob / total if total > 0 else (0.9 if output_letter == 'B' else 0.1)
        
        final_prediction = "Benefit" if confidence > 0.5 else "Hurt"
        
        if verbose:
            print(f"  {output_letter}: B={b_prob:.3f}, H={h_prob:.3f} → conf={confidence:.3f}")
        
        return {
            "prediction": final_prediction,
            "confidence": confidence,
            "b_prob": b_prob,
            "h_prob": h_prob,
            "success": True
        }
        
    except Exception as e:
        if verbose:
            print(f"  Error: {e}")
        return {
            "prediction": "Hurt",
            "confidence": 0.5,
            "error": str(e),
            "success": False
        }

# ============ 批量预测 ============
print("\n2. 运行 Few-shot 预测...")
print("-" * 80)

results = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Processing"):
    qid = row['qid']
    query = qid_to_query.get(qid, f"[Unknown QID {qid}]")
    
    top5 = df_colbert[df_colbert['qid'] == qid].nlargest(5, 'score')
    if len(top5) == 0:
        continue
    
    result = predict_few_shot(qid, query, top5, verbose=False)
    
    results.append({
        'qid': qid,
        'query': query,
        'true_b_pref_ratio': row['b_preference_ratio'],
        'true_preference': row['preference'],
        'llm_prediction': result['prediction'],
        'llm_confidence': result['confidence'],
        'b_prob': result.get('b_prob', 0),
        'h_prob': result.get('h_prob', 0),
        'success': result['success']
    })
    
    time.sleep(0.5)

results_df = pd.DataFrame(results)
results_df.to_csv('qwen_few_shot_predictions.csv', index=False)
print(f"\n✓ 完成: {len(results_df)} queries, {results_df['success'].sum()} successful")

# ============ 评估 ============
eval_results = results_df[results_df['success'] == True].copy()

print("\n" + "=" * 80)
print("EVALUATION RESULTS (Few-Shot)")
print("=" * 80)

# 相关性
tau, p_tau = kendalltau(eval_results['llm_confidence'], eval_results['true_b_pref_ratio'])
rho, p_rho = spearmanr(eval_results['llm_confidence'], eval_results['true_b_pref_ratio'])

print(f"\nKendall's Tau:  τ = {tau:.4f} (p = {p_tau:.4f})")
print(f"Spearman's Rho: ρ = {rho:.4f} (p = {p_rho:.4f})")

sig = "***" if p_tau < 0.001 else ("**" if p_tau < 0.01 else ("*" if p_tau < 0.05 else ""))
strength = ("Negligible" if abs(tau) < 0.1 else 
            ("Weak" if abs(tau) < 0.3 else 
             ("Moderate" if abs(tau) < 0.5 else "Strong")))
print(f"Correlation: {strength} {'Positive' if tau > 0 else 'Negative'} {sig}")

# 分类性能
eval_results['true_label'] = eval_results['true_b_pref_ratio'].apply(
    lambda x: 'Benefit' if x > 0.55 else ('Hurt' if x < 0.45 else 'Neutral')
)
clf_df = eval_results[eval_results['true_label'] != 'Neutral'].copy()

if len(clf_df) > 0:
    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(clf_df['true_label'], clf_df['llm_prediction'])
    f1 = f1_score(clf_df['true_label'], clf_df['llm_prediction'], pos_label='Benefit', zero_division=0)
    
    print(f"\nClassification:")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Score: {f1:.4f}")

# 预测分布
print(f"\nPrediction Distribution:")
print(f"  Predicted Benefit: {(eval_results['llm_prediction'] == 'Benefit').sum()}")
print(f"  Predicted Hurt:    {(eval_results['llm_prediction'] == 'Hurt').sum()}")

# 置信度分布
print(f"\nConfidence Statistics:")
print(f"  Mean: {eval_results['llm_confidence'].mean():.3f}")
print(f"  Std:  {eval_results['llm_confidence'].std():.3f}")
print(f"  Min:  {eval_results['llm_confidence'].min():.3f}")
print(f"  Max:  {eval_results['llm_confidence'].max():.3f}")

bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
print(f"\nConfidence Distribution:")
for interval, count in pd.cut(eval_results['llm_confidence'], bins=bins).value_counts().sort_index().items():
    print(f"  {interval}: {count}")

# 对比 Zero-shot（如果有）
print(f"\n" + "=" * 80)
print("COMPARISON")
print("=" * 80)

try:
    zero_shot_df = pd.read_csv('qwen_zero_shot_predictions_fixed.csv')
    zero_shot_valid = zero_shot_df[zero_shot_df['success'] == True]
    tau_zero, p_zero = kendalltau(zero_shot_valid['llm_confidence'], 
                                   zero_shot_valid['true_b_pref_ratio'])
    
    print(f"\n{'Method':<20} {'Kendall τ':<12} {'P-value':<12}")
    print("-" * 50)
    print(f"{'Zero-shot':<20} {tau_zero:<12.4f} {p_zero:<12.4f}")
    print(f"{'Few-shot (3 ex)':<20} {tau:<12.4f} {p_tau:<12.4f}")
    print("-" * 50)
    
    if abs(tau) > abs(tau_zero):
        improvement = ((abs(tau) - abs(tau_zero)) / abs(tau_zero) * 100) if tau_zero != 0 else float('inf')
        print(f"\n✅ Few-shot improves by {improvement:.1f}%")
    else:
        print(f"\n⚠️ Few-shot similar to or worse than zero-shot")
        
except FileNotFoundError:
    print("\nNo zero-shot results found for comparison")

print(f"\n📁 Output: qwen_few_shot_predictions.csv")
print("\n" + "=" * 80)

Few-Shot PRF Prediction with Qwen-2.5-72B

1. 加载数据...
✓ 加载了 40 个有效查询

2. 运行 Few-shot 预测...
--------------------------------------------------------------------------------


Processing: 100%|██████████| 40/40 [00:45<00:00,  1.14s/it]


✓ 完成: 40 queries, 40 successful

EVALUATION RESULTS (Few-Shot)

Kendall's Tau:  τ = 0.0386 (p = 0.7266)
Spearman's Rho: ρ = 0.0465 (p = 0.7759)
Correlation: Negligible Positive 

Classification:
  Accuracy: 0.5000
  F1-Score: 0.6486

Prediction Distribution:
  Predicted Benefit: 38
  Predicted Hurt:    2

Confidence Statistics:
  Mean: 0.930
  Std:  0.228
  Min:  0.000
  Max:  1.000

Confidence Distribution:
  (0.0, 0.2]: 2
  (0.2, 0.4]: 0
  (0.4, 0.6]: 1
  (0.6, 0.8]: 0
  (0.8, 1.0]: 37

COMPARISON

Method               Kendall τ    P-value     
--------------------------------------------------
Zero-shot            -0.2211      0.0450      
Few-shot (3 ex)      0.0386       0.7266      
--------------------------------------------------

⚠️ Few-shot similar to or worse than zero-shot

📁 Output: qwen_few_shot_predictions.csv

